# Notebook 3 -- A surrogate model for the LLMs' votes

**Question.** How much of an LLM's vote can be reconstructed from interpretable
features of the dialogue -- and does it matter *who a persuasion act was aimed
at and what it claimed*, or only that the act occurred?

**Target.** The 3 stochastic runs of a game are pooled into one instance whose
target is the empirical vote distribution over the ballot. If the LLM voted
Alice twice and Bob once, the target is Alice 2/3, Bob 1/3. Split games stay
split instead of becoming three contradictory hard labels.

**Ballot.** The roster plus **"No Werewolf"**, which these LLMs do choose (6-23%
of votes depending on the model), entering as one alternative with only an
intercept.

**Feature blocks -- the same persuasion acts at two resolutions.**

| block | what it knows |
|---|---|
| **A -- techniques** | the annotated act was *used*: per-player counts of Accusation, Defense, Interrogation, Identity Declaration, Evidence, Call for Action, plus utterance count |
| **B -- resolved** | the same acts with their target and content: who an accusation *landed on* (werewolf-/deception-type accusations received) and what an identity claim *said* (claims Werewolf, claims an info role, number of distinct roles claimed) |
| **C** | A + B |

Each also has a **temporal** variant splitting the same features into the first
and second half of the game. Consistency judgements (role conflicts,
self-contradictions) are excluded: they are verdicts on whether a claim held
up, not persuasion acts.

**Model.** McFadden's conditional logit -- the standard estimator for choosing
one alternative from a set of varying size -- unpenalised, ridge and lasso,
plus gradient boosting as a non-linear reference. Scored by held-out
log-likelihood, McFadden's pseudo-R-squared and hit rate, under 5-fold CV
grouped by game x 3 seeds.

The machinery lives in `src/utils_choice` and is shared with the human-vote
notebook, so the two analyses differ only in whose votes they explain.

## Setup

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd


def find_repo_root(start=None, repo_name="masters_thesis_sdg"):
    current = (start or Path.cwd()).resolve()
    while True:
        if current.name == repo_name:
            return current
        if current.parent == current:
            raise FileNotFoundError(repo_name)
        current = current.parent


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))

from utils_choice import (BLOCKS, CIRCLE_OPTION, PLAYER_FEATURES, BLOCK_A,
                          BLOCK_B, build_ballot, build_frame, cols_for,
                          coef_table, block_test, block_bootstrap, cross_validate,
                          iia_test, llm_vote_targets, load_accusation_features,
                          load_identity_claim_features, load_technique_features,
                          load_vote_tables, reference_points, run_grid,
                          run_validation_checks, save_tables, shares_from_targets,
                          stability_selection, crowd_modal_map,
                          STOCHASTIC_RUNS, GREEDY_RUN, STABILITY_THRESHOLD)

MODEL_STAGE, PROMPT_DIR = "base", "prompt_v4"
ANALYSIS_ROOT = REPO_ROOT / "analysis"
TABLES_REL = Path(MODEL_STAGE) / "voting" / PROMPT_DIR / "vote_stability" / "tables"
ANNOT_ROOT = REPO_ROOT / "data" / "raw" / "lai2023"
ACC_ROOT = (REPO_ROOT / "data" / "processed" / "lai2023"
            / "accusation_transcripts" / "acc_targets")
IC_CSV = (REPO_ROOT / "data" / "processed" / "lai2023"
          / "identity_claim_transcripts" / "ic_targets"
          / "player_conflict_features.csv")
VILLAGE_CSV = (ANALYSIS_ROOT / "human_outcomes" / "vote_tables"
               / "village_vote_dispersion.csv")
OUT_DIR = ANALYSIS_ROOT / "cross_model" / MODEL_STAGE / "voting" / PROMPT_DIR / "predictive" / "tables"

for msg in run_validation_checks():
    print("estimator validated:", msg)

estimator validated: recovers a known beta from simulated choices (max |error| = 0.018)
estimator validated: matches scikit-learn on the J=2 reduction (max |difference| = 1.6e-04)
estimator validated: analytic gradient matches the numerical one (error = 5.2e-05)
estimator validated: choice probabilities sum to 1 in every set (max error = 2.2e-16)
estimator validated: a strong L1 penalty zeroes coefficients (1 of 6 survive)


## Features and ballot

In [2]:
votes, games, roster_by_key = load_vote_tables(ANALYSIS_ROOT, TABLES_REL)
pt_df, game_length, rec_speaker, annotated = load_technique_features(ANNOT_ROOT)
acc_counts, n_acc = load_accusation_features(ACC_ROOT, game_length)
ic_feats = load_identity_claim_features(IC_CSV, game_length, rec_speaker)
ballot = build_ballot(roster_by_key, pt_df, acc_counts, ic_feats)

print(f"games with rosters: {len(roster_by_key)}; annotated games matching one: "
      f"{sum(1 for k in annotated if k in roster_by_key)}")
print(f"accusation events: {n_acc}; identity-claim rows: {len(ic_feats)}")
print(f"ballot rows: {len(ballot)} over {ballot['key'].nunique()} games "
      "(players + one 'No Werewolf' alternative each)")

players_only = ballot[ballot["is_circle_option"] == 0]
display(pd.DataFrame({
    "block": ["A"] * len(BLOCK_A) + ["B"] * len(BLOCK_B),
    "share_nonzero": [(players_only[c] > 0).mean().round(3) for c in BLOCK_A + BLOCK_B],
    "mean": [players_only[c].mean().round(2) for c in BLOCK_A + BLOCK_B]},
    index=BLOCK_A + BLOCK_B))
print(f"plus early/late variants of each ({len(PLAYER_FEATURES)} columns in total)")

games with rosters: 191; annotated games matching one: 191
accusation events: 2001; identity-claim rows: 632
ballot rows: 1055 over 191 games (players + one 'No Werewolf' alternative each)


,block,share_nonzero,mean
pt_accusation,A,0.854,3.98
pt_defense,A,0.816,3.73
pt_interrogation,A,0.914,4.67
pt_identity_declaration,A,0.728,1.54
pt_evidence,A,0.832,2.55
pt_call_for_action,A,0.631,1.59
n_utterances,A,0.978,27.40
werewolf_count,B,0.527,1.36
deception_count,B,0.416,0.92
claims_werewolf,B,0.110,0.11


plus early/late variants of each (36 columns in total)


## Targets

One instance per (LLM, game): the vote distribution over the ballot. The greedy
(T=0) vote is kept as a separate target for the robustness check.

In [3]:
TARGETS, summary, unmatched = llm_vote_targets(votes, roster_by_key, STOCHASTIC_RUNS)
GREEDY, _, _ = llm_vote_targets(votes, roster_by_key, [GREEDY_RUN])
MODELS = sorted(TARGETS)
print("models:", MODELS, "| runs dropped for unmatched names:", unmatched)
display(summary.groupby("model").agg(games=("key", "nunique"),
                                     mean_runs=("n_runs", "mean"),
                                     share_split=("is_split", "mean"),
                                     circle_share=("circle_share", "mean")).round(3))

frames = {m: build_frame(TARGETS[m], ballot) for m in MODELS}

models: ['2B', '31B', '4B'] | runs dropped for unmatched names: {'2B': 0, '31B': 0, '4B': 0}


,games,mean_runs,share_split,circle_share
model,,,,
2B,191,2.974,0.529,0.229
31B,191,3.000,0.382,0.120
4B,191,3.000,0.466,0.059


## Results

The whole grid is reported: the A-vs-B contrast *is* the research question, not
a nuisance to be optimised away. Reference points sit alongside -- the
equal-shares null, and the ceiling set by the LLM's own test-retest
reliability, which is what the hit rate should be read against rather than 1.0.

In [4]:
results = pd.concat([run_grid(frames[m], BLOCKS, label=m) for m in MODELS],
                    ignore_index=True)
crowd = crowd_modal_map(pd.read_csv(VILLAGE_CSV))
references = pd.concat(
    [reference_points(frames[m], shares_from_targets(TARGETS[m]), label=m,
                      crowd_modal=crowd) for m in MODELS], ignore_index=True)

print("McFadden pseudo-R^2 (conditional logit only; gbm is not a choice model):")
display(results.pivot_table(index="model", columns=["block", "learner"],
                            values="mcfadden_r2").round(3))
print("\nHit rate:")
display(results.pivot_table(index="model", columns=["block", "learner"],
                            values="hit_rate").round(3))
print("\nReference points:")
display(references.round(3))

McFadden pseudo-R^2 (conditional logit only; gbm is not a choice model):


block   A_techniques                     A_temporal                      \
learner       clogit clogit_l1 clogit_l2     clogit clogit_l1 clogit_l2   
model                                                                     
2B             0.009     0.008     0.011      0.010     0.019     0.021   
31B            0.004     0.003     0.009     -0.002     0.000     0.012   
4B             0.045     0.043     0.049      0.049     0.056     0.061   

block   B_resolved                     B_temporal                     C_both  \
learner     clogit clogit_l1 clogit_l2     clogit clogit_l1 clogit_l2 clogit   
model                                                                          
2B           0.180     0.180     0.180      0.169     0.174     0.168  0.186   
31B          0.163     0.162     0.160      0.152     0.154     0.154  0.161   
4B           0.210     0.209     0.211      0.203     0.204     0.207  0.196   

block                       C_temporal                      
learner clogit_l1 clogit_l2     clogit clogit_l1 clogit_l2  
model                                                       
2B          0.189     0.188      0.153     0.169     0.168  
31B         0.159     0.160      0.128     0.144     0.146  
4B          0.207     0.196      0.188     0.201     0.203


Hit rate:


block   A_techniques                            A_temporal            \
learner       clogit clogit_l1 clogit_l2    gbm     clogit clogit_l1   
model                                                                  
2B             0.241     0.240     0.236  0.215      0.262     0.254   
31B            0.230     0.208     0.240  0.216      0.237     0.218   
4B             0.273     0.248     0.265  0.260      0.312     0.311   

block                    B_resolved            ... B_temporal        C_both  \
learner clogit_l2    gbm     clogit clogit_l1  ...  clogit_l2    gbm clogit   
model                                          ...                            
2B          0.252  0.251      0.474     0.469  ...      0.497  0.470  0.493   
31B         0.244  0.209      0.429     0.427  ...      0.411  0.400  0.434   
4B          0.304  0.301      0.486     0.484  ...      0.501  0.482  0.470   

block                              C_temporal                             
learner clogit_l1 clogit_l2    gbm     clogit clogit_l1 clogit_l2    gbm  
model                                                                     
2B          0.493     0.495  0.475      0.496     0.478     0.483  0.465  
31B         0.431     0.430  0.407      0.421     0.422     0.417  0.379  
4B          0.486     0.472  0.449      0.471     0.490     0.485  0.464  

[3 rows x 24 columns]


Reference points:


,model,null_ll,null_hit_rate,ceiling_agreement,ceiling_best_possible,irreducible_entropy,most_talkative_hit,most_accused_hit,crowd_modal_hit
0,2B,-1.698,0.185,0.741,0.787,0.386,0.145,0.361,0.358
1,31B,-1.699,0.184,0.815,0.850,0.275,0.187,0.401,0.429
2,4B,-1.699,0.184,0.773,0.815,0.338,0.213,0.402,0.404


## The pre-specified model

Interpretation is read off **one configuration, fixed in advance**: the full
feature set `C_both` with the ridge penalty, the same for every LLM. Nothing is
chosen by looking at the numbers being reported, so the coefficients and their
intervals carry no selection bias.

The full set is interpreted deliberately: no feature has been selected away, so
every feature gets a coefficient -- including the null ones, which is itself
part of the result. Whether a block earns its place is settled by the
bootstrapped comparison below, not by which model gets interpreted.

Coefficients are log-odds *within a ballot*: a conditional logit has no
intercept and anything constant across a roster cancels, so each one already
means "relative to the other players in this game".

In [5]:
PRIMARY_BLOCK, PRIMARY_LEARNER = "C_both", "clogit_l2"
PRIMARY_COLS = cols_for(PRIMARY_BLOCK)

best_rows, perm_rows, coef_rows = [], [], []
for m in MODELS:
    grid = (results[(results["model"] == m) & (results["learner"] != "gbm")]
            .sort_values("mcfadden_r2", ascending=False).reset_index(drop=True))
    hit = grid[(grid["block"] == PRIMARY_BLOCK) & (grid["learner"] == PRIMARY_LEARNER)]
    row, rank = hit.iloc[0], int(hit.index[0]) + 1
    best_rows.append({"model": m, "block": PRIMARY_BLOCK, "learner": PRIMARY_LEARNER,
                      "mcfadden_r2": row["mcfadden_r2"], "hit_rate": row["hit_rate"],
                      "rank_in_grid": f"{rank} of {len(grid)}",
                      "cost_of_prespecifying": round(
                          grid.iloc[0]["mcfadden_r2"] - row["mcfadden_r2"], 3)})

    _, perm = cross_validate(frames[m], PRIMARY_COLS, PRIMARY_LEARNER,
                             collect_perm=True)
    for f, d in perm.items():
        perm_rows.append({"model": m, "feature": f,
                          "ll_drop": round(float(np.mean(d)), 4),
                          "share_folds_positive": round(float(np.mean(np.array(d) > 0)), 2)})

    tab, _, n_cl = coef_table(frames[m], PRIMARY_COLS)
    tab.insert(0, "model", m)
    coef_rows.append(tab)

best_config = pd.DataFrame(best_rows)
permutation_importance = pd.DataFrame(perm_rows)
coefficients = pd.concat(coef_rows, ignore_index=True)

print(f"Pre-specified configuration ({PRIMARY_BLOCK} / {PRIMARY_LEARNER}):")
display(best_config)
for m in MODELS:
    print(f"\n{m} -- odds ratios per 1 SD, cluster-robust by game:")
    display(coefficients[coefficients["model"] == m].drop(columns="model")
            .sort_values("p").round({"odds_ratio": 2, "ci_lo": 2, "ci_hi": 2,
                                     "z": 2, "p": 4, "p_bonferroni": 3})
            .reset_index(drop=True))
print("\nHeld-out permutation importance (drop in log-likelihood when shuffled):")
display(permutation_importance.pivot_table(index="feature", columns="model",
                                           values="ll_drop")
        .loc[PRIMARY_COLS].round(3))

Pre-specified configuration (C_both / clogit_l2):


,model,block,learner,mcfadden_r2,hit_rate,rank_in_grid,cost_of_prespecifying
0,2B,C_both,clogit_l2,0.188,0.495,2 of 18,0.001
1,31B,C_both,clogit_l2,0.160,0.430,5 of 18,0.003
2,4B,C_both,clogit_l2,0.196,0.472,10 of 18,0.015



2B -- odds ratios per 1 SD, cluster-robust by game:


,feature,odds_ratio,ci_lo,ci_hi,z,p,p_bonferroni
0,werewolf_count,2.17,1.76,2.69,7.14,0.0000,0.000
1,claims_werewolf,1.79,1.50,2.13,6.58,0.0000,0.000
2,n_utterances,0.48,0.32,0.74,-3.38,0.0007,0.010
3,pt_defense,1.46,1.16,1.83,3.26,0.0011,0.015
4,is_circle_option,1.32,1.11,1.57,3.13,0.0018,0.023
5,pt_accusation,1.36,1.08,1.70,2.66,0.0078,0.101
6,pt_interrogation,1.12,0.90,1.40,1.00,0.3166,1.000
7,pt_identity_declaration,0.88,0.69,1.14,-0.96,0.3383,1.000
8,claims_info_role,0.92,0.74,1.15,-0.75,0.4552,1.000
9,deception_count,1.05,0.91,1.22,0.71,0.4769,1.000



31B -- odds ratios per 1 SD, cluster-robust by game:


,feature,odds_ratio,ci_lo,ci_hi,z,p,p_bonferroni
0,werewolf_count,2.06,1.68,2.53,6.98,0.0000,0.000
1,claims_werewolf,1.37,1.16,1.61,3.68,0.0002,0.003
2,deception_count,1.34,1.11,1.62,3.08,0.0020,0.027
3,claims_info_role,0.79,0.63,0.98,-2.10,0.0357,0.464
4,pt_defense,1.28,1.01,1.64,2.02,0.0437,0.569
5,pt_accusation,1.26,0.99,1.59,1.89,0.0587,0.763
6,n_utterances,0.66,0.43,1.02,-1.88,0.0606,0.788
7,n_distinct_roles_claimed_self,1.23,0.91,1.66,1.34,0.1799,1.000
8,pt_call_for_action,1.16,0.93,1.44,1.32,0.1881,1.000
9,pt_identity_declaration,0.85,0.67,1.09,-1.26,0.2075,1.000



4B -- odds ratios per 1 SD, cluster-robust by game:


,feature,odds_ratio,ci_lo,ci_hi,z,p,p_bonferroni
0,claims_werewolf,1.92,1.63,2.27,7.82,0.0000,0.000
1,werewolf_count,1.64,1.33,2.02,4.66,0.0000,0.000
2,deception_count,1.21,1.02,1.45,2.15,0.0313,0.407
3,is_circle_option,0.81,0.64,1.02,-1.78,0.0749,0.973
4,pt_defense,1.20,0.92,1.55,1.36,0.1723,1.000
5,n_distinct_roles_claimed_self,0.90,0.66,1.22,-0.68,0.4980,1.000
6,claims_info_role,0.94,0.77,1.14,-0.65,0.5144,1.000
7,pt_evidence,0.93,0.74,1.18,-0.60,0.5517,1.000
8,pt_interrogation,0.95,0.77,1.17,-0.48,0.6286,1.000
9,pt_accusation,1.05,0.82,1.34,0.35,0.7244,1.000



Held-out permutation importance (drop in log-likelihood when shuffled):


model,2B,31B,4B
feature,,,
pt_accusation,0.032,0.011,-0.004
pt_defense,0.070,0.015,0.011
pt_interrogation,0.002,0.000,-0.002
pt_identity_declaration,0.003,-0.002,-0.001
pt_evidence,-0.003,0.006,-0.004
pt_call_for_action,0.001,0.009,-0.001
n_utterances,0.183,0.032,-0.001
werewolf_count,0.448,0.317,0.135
deception_count,-0.000,0.057,0.018


## Inference

Cross-validation spread is not a significance test. Three tools: **nested block
tests** (robust Wald on what one block adds to another -- Wald rather than a
likelihood ratio, which is invalid under a sandwich covariance); **stability
selection** (the lasso refit on 200 half-samples of games at a penalty tuned to
keep about five features, because at the CV-optimal penalty nothing is dropped
at this n/p ratio); and a **game-level bootstrap** of the out-of-sample block
differences.

The Wald test asks whether a coefficient is reliably non-zero; stability
selection asks whether a feature survives when the model must be sparse. Report
a feature as a finding when both agree, and say so explicitly when they differ.

In [6]:
block_rows, stab_rows, boot_rows = [], [], []
COMPARISONS = [("A_techniques", "B_resolved"), ("B_resolved", "C_both"),
               ("A_techniques", "C_both"), ("A_techniques", "A_temporal"),
               ("B_resolved", "B_temporal"), ("C_both", "C_temporal")]
for m in MODELS:
    for small, label in [("A_techniques", "B_resolved adds over A_techniques"),
                         ("B_resolved", "A_techniques adds over B_resolved")]:
        W, dfree, p = block_test(frames[m], small)
        block_rows.append({"model": m, "test": label, "chi2": round(W, 1),
                           "df": dfree, "p_value": p})
    tab, _ = stability_selection(frames[m], PRIMARY_COLS)
    tab.insert(0, "model", m)
    stab_rows.append(tab)
    null_ll = -references.set_index("model").loc[m, "null_ll"]
    boot_rows.append(block_bootstrap(frames[m], COMPARISONS, null_ll, label=m))

block_tests = pd.DataFrame(block_rows)
stability = pd.concat(stab_rows, ignore_index=True)
bootstrap = pd.concat(boot_rows, ignore_index=True)

print("Nested block tests (robust Wald):")
display(block_tests)
print(f"\nStability selection (share of half-samples keeping the feature; "
      f">= {STABILITY_THRESHOLD} is the conventional threshold):")
display(stability.pivot_table(index="feature", columns="model",
                              values="selection_freq").loc[PRIMARY_COLS].round(2))
print("\nGame-level bootstrap on held-out log-likelihood:")
display(bootstrap)

Nested block tests (robust Wald):


,model,test,chi2,df,p_value
0,2B,B_resolved adds over A_techniques,107.2,5,1.619586e-21
1,2B,A_techniques adds over B_resolved,20.3,7,4.916401e-03
2,31B,B_resolved adds over A_techniques,104.9,5,4.821757e-21
3,31B,A_techniques adds over B_resolved,10.9,7,1.435998e-01
4,4B,B_resolved adds over A_techniques,102.9,5,1.281400e-20
5,4B,A_techniques adds over B_resolved,2.7,7,9.144976e-01



Stability selection (share of half-samples keeping the feature; >= 0.6 is the conventional threshold):


model,2B,31B,4B
feature,,,
pt_accusation,0.06,0.02,0.18
pt_defense,0.28,0.14,0.48
pt_interrogation,0.09,0.10,0.16
pt_identity_declaration,0.10,0.01,0.22
pt_evidence,0.14,0.11,0.20
pt_call_for_action,0.11,0.06,0.14
n_utterances,0.20,0.01,0.02
werewolf_count,1.00,1.00,1.00
deception_count,0.14,0.90,0.88



Game-level bootstrap on held-out log-likelihood:


,model,comparison,delta_mean_ll,ci_lo,ci_hi,delta_pseudo_r2,excludes_zero
0,2B,B_resolved - A_techniques,0.2855,0.1909,0.3790,0.1681,True
1,2B,C_both - B_resolved,0.0125,-0.0157,0.0403,0.0073,False
2,2B,C_both - A_techniques,0.2979,0.2024,0.3912,0.1754,True
3,2B,A_temporal - A_techniques,0.0171,-0.0049,0.0388,0.0101,False
4,2B,B_temporal - B_resolved,-0.0218,-0.0555,0.0089,-0.0128,False
5,2B,C_temporal - C_both,-0.0334,-0.0706,0.0025,-0.0197,False
6,31B,B_resolved - A_techniques,0.2576,0.1791,0.3373,0.1516,True
7,31B,C_both - B_resolved,-0.0011,-0.0233,0.0213,-0.0006,False
8,31B,C_both - A_techniques,0.2565,0.1813,0.3360,0.1509,True
9,31B,A_temporal - A_techniques,0.0045,-0.0183,0.0271,0.0027,False


## Robustness

Does the finding depend on sampling temperature, and does the model's IIA
assumption hold? IIA is what a conditional logit rests on and the standard
objection to it: the relative odds of two players should not depend on who else
is on the ballot. The Hausman-McFadden test drops the "No Werewolf"
alternative, which is also the substantively interesting restriction.

In [7]:
greedy_frames = {m: build_frame(GREEDY[m], ballot) for m in MODELS}
greedy_check = pd.concat(
    [run_grid(greedy_frames[m], BLOCKS, learners=["clogit_l2"], label=m)
     for m in MODELS], ignore_index=True)
print("Same grid on the deterministic greedy (T=0) vote:")
display(greedy_check.pivot_table(index="model", columns="block",
                                 values=["mcfadden_r2", "hit_rate"]).round(3))

iia_rows, iia_tabs = [], []
for m in MODELS:
    summary_m, tab = iia_test(frames[m], PRIMARY_COLS)
    summary_m.insert(0, "model", m); tab.insert(0, "model", m)
    iia_rows.append(summary_m); iia_tabs.append(tab)
iia_summary = pd.concat(iia_rows, ignore_index=True)
iia_coefficients = pd.concat(iia_tabs, ignore_index=True)
print("\nIIA (Hausman-McFadden, dropping the 'No Werewolf' alternative):")
display(iia_summary)

Same grid on the deterministic greedy (T=0) vote:


hit_rate                                                     \
block A_techniques A_temporal B_resolved B_temporal C_both C_temporal   
model                                                                   
2B           0.246      0.274      0.512      0.475  0.494      0.463   
31B          0.232      0.251      0.363      0.366  0.349      0.363   
4B           0.323      0.371      0.499      0.494  0.476      0.470   

       mcfadden_r2                                                     
block A_techniques A_temporal B_resolved B_temporal C_both C_temporal  
model                                                                  
2B           0.009      0.013      0.193      0.165  0.188      0.152  
31B          0.006      0.016      0.081      0.078  0.070      0.075  
4B           0.053      0.057      0.222      0.214  0.217      0.195


IIA (Hausman-McFadden, dropping the 'No Werewolf' alternative):


,model,hausman_chi2,df,p_value,verdict,max_abs_log_shift
0,2B,-26.89,12,NaN,inconclusive (statistic not positive - a known...,0.108
1,31B,42.94,12,0.000,IIA rejected,0.111
2,4B,9.08,12,0.696,no evidence against IIA,0.049


In [8]:
written, removed = save_tables(OUT_DIR, {
    "surrogate_results": results,
    "surrogate_reference_points": references,
    "surrogate_best_config": best_config,
    "surrogate_coefficients": coefficients,
    "surrogate_permutation_importance": permutation_importance,
    "surrogate_block_tests": block_tests,
    "surrogate_stability_selection": stability,
    "surrogate_block_bootstrap": bootstrap,
    "surrogate_greedy_robustness": greedy_check,
    "surrogate_iia_test": iia_summary,
    "surrogate_iia_coefficients": iia_coefficients,
})
print("saved ->", OUT_DIR.relative_to(REPO_ROOT))
for n in written:
    print("  ", n)
if removed:
    print("removed stale files from superseded versions:")
    for n in removed:
        print("  ", n)

saved -> analysis\cross_model\base\voting\prompt_v4\predictive\tables
   surrogate_best_config.csv
   surrogate_block_bootstrap.csv
   surrogate_block_tests.csv
   surrogate_coefficients.csv
   surrogate_greedy_robustness.csv
   surrogate_iia_coefficients.csv
   surrogate_iia_test.csv
   surrogate_permutation_importance.csv
   surrogate_reference_points.csv
   surrogate_results.csv
   surrogate_stability_selection.csv


## Reading guide

- **McFadden's pseudo-R-squared is the headline.** It is not an R-squared in
  the linear sense; for choice models 0.2-0.4 is conventionally a good fit.
- **The hit rate's ceiling is well below 1.** `ceiling_agreement` is the chance
  two independent runs of the same LLM agree -- the reliability of the thing
  being predicted. Judge the model by the distance to that, not to 1.0, and
  read `irreducible_entropy` as the same limit in log-likelihood units.
- **A vs B vs C is the research question**, so the whole grid is reported. For
  block comparisons use the bootstrap intervals, not the fold spread: the 3
  seeds reshuffle the same games and understate uncertainty.
- **The interpreted configuration is fixed in advance**, so neither the quoted
  pseudo-R-squared nor the coefficient intervals are contaminated by having
  picked a winner. Its rank in the grid is printed.
- **`gbm` is not a choice model** -- a pointwise ranker included to check
  whether non-linearity buys anything. Only its hit rate is comparable.
- p-values are Bonferroni-corrected within each model; permutation importance
  splits credit arbitrarily between correlated features.
- Everything is correlational: features that track the vote, not causes of it.